[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/25_flash_attention.ipynb)

# 🔴 Hard: Flash Attention (Tiled)

Implement **tiled attention with online softmax** — the core idea behind Flash Attention.

### Signature
```python
def flash_attention(Q, K, V, block_size=32) -> Tensor:
    # Q, K, V: (B, S, D)
    # Returns: (B, S, D) — same as standard attention
```

### Key Insight
Instead of materializing the full S×S attention matrix, process in blocks:
1. For each Q-block, iterate over K/V blocks
2. Use **online softmax**: track running `max` and `sum`
3. Rescale accumulator when max changes: `acc *= exp(old_max - new_max)`
4. Final: `output = acc / row_sum`

Must give **identical** results to standard softmax attention.

In [1]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 1.4 MB/s eta 0:00:00


In [2]:
import torch
import math

In [34]:
# ✏️ YOUR IMPLEMENTATION HERE

def flash_attention(Q, K, V, block_size=32):
    # Process Q in blocks, iterate K/V blocks with online softmax
    B, N, d = Q.shape
    Tc = block_size
    T = N / Tc
    m = torch.full((B,N,1), float('-inf'))
    l = torch.zeros(B,N,1)
    o = torch.zeros(B,N,d)
    for i in range(0,N,Tc):
      s = torch.einsum('bov,biv->boi', Q, K[:,i:i+Tc,:]) * math.sqrt(d)**(-1)
      m_new = torch.maximum(torch.max(s, dim=-1)[0].unsqueeze(2), m)
      p = torch.exp(s - m_new) # boi, boi
      alpha = torch.exp(m - m_new)
      p = torch.exp(s - m_new) # boi, bo1
      # print(p.shape, torch.sum(p, dim=-1, keepdim=True).shape)
      # print(s.shape, m_new.shape, p.shape)
      l_new = l * alpha + torch.sum(p, dim=-1, keepdim=True)
      o_new = o * alpha + torch.einsum('boi,biv->bov', p, V[:, i:i+Tc,:])
      m = m_new
      l = l_new
      o = o_new
    return o/l
    # pass

In [35]:
# 🧪 Debug
import math
Q, K, V = torch.randn(1, 8, 4), torch.randn(1, 8, 4), torch.randn(1, 8, 4)
out = flash_attention(Q, K, V, block_size=4)
scores = torch.bmm(Q, K.transpose(1,2)) / math.sqrt(4)
ref = torch.bmm(torch.softmax(scores, dim=-1), V)
print('Match:', torch.allclose(out, ref, atol=1e-4))

Match: True


In [36]:
# ✅ SUBMIT
from torch_judge import check
check('flash_attention')


🧪 Testing: Flash Attention (Tiled) (Hard)
──────────────────────────────────────────────────
  ✅ [1/4] Matches standard attention (32.3ms)
  ✅ [2/4] Non-aligned block size (2.6ms)
  ✅ [3/4] Block size invariant (3.3ms)
  ✅ [4/4] Gradient flow (41.3ms)
──────────────────────────────────────────────────
  🎉 All 4 tests passed! (79.4ms total)
  Progress saved. Run status() to see your dashboard.

